# J3 — Analytics SQL & Couche Gold (Olist)

Notebook autonome à exécuter **après J2**. Il lit `/lakehouse/silver/orders_enriched`, crée quatre tables Gold Delta et contrôle leurs résultats.

**Pré-requis :** NameNode/HDFS opérationnel, table Silver déjà écrite. Après une erreur `ConnectionRefusedError`, redémarrer le kernel Jupyter avant de lancer ce notebook. Ne pas lancer J2 en parallèle pour limiter les conflits de mémoire/ports.

## 1 — Spark et chemins HDFS

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType

# Utilise la configuration Spark/Delta déjà fournie par le conteneur Jupyter.
# Ne pas appeler spark.stop() dans ce notebook.
spark = SparkSession.builder.appName("J3_Analytics_SQL_Gold").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.shuffle.partitions", "16")
spark.conf.set("spark.sql.adaptive.enabled", "true")

HDFS = "hdfs://namenode:9000"
SILVER_MAIN_PATH = f"{HDFS}/lakehouse/silver/orders_enriched"
GOLD_ROOT = f"{HDFS}/lakehouse/gold"

print("Spark :", spark.version)
print("Spark UI :", spark.sparkContext.uiWebUrl)
print("Test Spark :", spark.range(5).count())

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-108c3c38-b8d8-48cf-9b5d-4b3a22336cbd;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found

Spark : 4.2.0
Spark UI : http://spark-jupyter:4043


Test Spark : 5


## 2 — Lecture Silver et vérifications

In [ ]:
silver = spark.read.format("delta").load(SILVER_MAIN_PATH)
silver.createOrReplaceTempView("v_silver")

required = {
    "order_id", "customer_unique_id", "customer_state", "product_category",
    "order_value", "item_count", "delivery_days", "delay_days", "is_late",
    "purchase_month", "order_status"
}
missing = sorted(required - set(silver.columns))
if missing:
    raise ValueError(f"Colonnes Silver manquantes : {missing}. Vérifier J2.")

silver_count = silver.count()
unique_orders = silver.select("order_id").distinct().count()
print(f"Silver : {silver_count:,} commandes | IDs distincts : {unique_orders:,}")
assert silver_count == unique_orders, "La table Silver contient des order_id dupliqués."
if silver_count != 99441:
    print("ATTENTION : nombre de commandes différent des 99 441 attendues pour Olist.")
silver.select("order_id", "purchase_month", "customer_state", "order_value", "review_score").show(5, truncate=False)

26/11/21 16:55:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 5:>                                                         (0 + 4) / 50]

## 3 — Préparation des colonnes analytiques

Les commandes peuvent avoir plusieurs catégories de produits. La table Silver J2 conserve une catégorie représentative par commande : le chiffre d’affaires par catégorie ci-dessous est donc une **attribution par commande**, pas une ventilation exacte au niveau de chaque article.

In [ ]:
base = (
    silver
    .withColumn("order_value_num", F.expr("try_cast(order_value AS DOUBLE)"))
    .withColumn("delivery_days_num", F.expr("try_cast(delivery_days AS DOUBLE)"))
    .withColumn("delay_days_num", F.expr("try_cast(delay_days AS DOUBLE)"))
    .withColumn("is_late_num", F.expr("try_cast(is_late AS INT)"))
    .withColumn("review_score_num", F.expr("try_cast(review_score AS DOUBLE)"))
    .withColumn("product_category_clean", F.coalesce(F.col("product_category"), F.lit("unknown")))
    .withColumn("customer_state_clean", F.coalesce(F.col("customer_state"), F.lit("unknown")))
)
base.createOrReplaceTempView("v_analytics")
print("Vue v_analytics disponible :", base.count(), "lignes")

## 4 — Gold : performance par catégorie

In [ ]:
category_performance = spark.sql("""
SELECT
    product_category_clean AS product_category,
    COUNT(*) AS total_orders,
    ROUND(SUM(COALESCE(order_value_num, 0)), 2) AS total_revenue,
    ROUND(AVG(order_value_num), 2) AS avg_order_value,
    ROUND(AVG(review_score_num), 2) AS avg_review_score,
    ROUND(AVG(delivery_days_num), 2) AS avg_delivery_days
FROM v_analytics
GROUP BY product_category_clean
""")
category_performance.orderBy(F.desc("total_revenue")).show(15, truncate=False)

## 5 — Gold : retards par État

Le taux de retard est calculé **uniquement sur les commandes dont la date réelle de livraison est disponible**. Les commandes non livrées ne sont pas considérées comme ponctuelles.

In [ ]:
late_orders_by_state = spark.sql("""
SELECT
    customer_state_clean AS customer_state,
    COUNT(*) AS total_orders,
    SUM(CASE WHEN delay_days_num IS NOT NULL THEN 1 ELSE 0 END) AS delivered_orders,
    SUM(CASE WHEN delay_days_num > 0 THEN 1 ELSE 0 END) AS late_orders,
    ROUND(100.0 * SUM(CASE WHEN delay_days_num > 0 THEN 1 ELSE 0 END)
        / NULLIF(SUM(CASE WHEN delay_days_num IS NOT NULL THEN 1 ELSE 0 END), 0), 2) AS late_rate_pct,
    ROUND(AVG(delay_days_num), 2) AS avg_delay_days,
    ROUND(AVG(delivery_days_num), 2) AS avg_delivery_days
FROM v_analytics
GROUP BY customer_state_clean
""")
late_orders_by_state.orderBy(F.desc("late_rate_pct")).show(30, truncate=False)

## 6 — Gold : évolution mensuelle des livraisons

In [ ]:
monthly_delivery = spark.sql("""
SELECT
    purchase_month,
    COUNT(*) AS total_orders,
    SUM(CASE WHEN delivery_days_num IS NOT NULL THEN 1 ELSE 0 END) AS delivered_orders,
    SUM(CASE WHEN delay_days_num > 0 THEN 1 ELSE 0 END) AS late_orders,
    ROUND(100.0 * SUM(CASE WHEN delay_days_num > 0 THEN 1 ELSE 0 END)
        / NULLIF(SUM(CASE WHEN delay_days_num IS NOT NULL THEN 1 ELSE 0 END), 0), 2) AS late_rate_pct,
    ROUND(AVG(delivery_days_num), 2) AS avg_delivery_days,
    ROUND(SUM(COALESCE(order_value_num, 0)), 2) AS total_revenue
FROM v_analytics
WHERE purchase_month IS NOT NULL
GROUP BY purchase_month
""")
monthly_delivery.orderBy("purchase_month").show(40, truncate=False)

## 7 — Gold : segmentation des clients (RFM simplifiée)

Les segments sont fondés sur le nombre de commandes et la dépense totale par client, sans seuil arbitraire de récence. La table conserve une ligne par `customer_unique_id`.

In [ ]:
customer_stats = (
    base.filter(F.col("customer_unique_id").isNotNull())
    .groupBy("customer_unique_id")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.round(F.sum(F.coalesce("order_value_num", F.lit(0.0))), 2).alias("total_spent"),
        F.max("purchase_date").alias("last_purchase_date")
    )
)
customer_segments = (
    customer_stats
    .withColumn(
        "segment",
        F.when(F.col("total_orders") >= 3, "loyal_3plus")
         .when(F.col("total_orders") == 2, "repeat_2")
         .otherwise("one_time")
    )
)
customer_segments.groupBy("segment").agg(
    F.count("*").alias("customers"),
    F.round(F.sum("total_spent"), 2).alias("revenue")
).orderBy("segment").show(truncate=False)

## 8 — Contrôles avant écriture Gold

In [ ]:
gold_tables = {
    "category_performance": category_performance,
    "late_orders_by_state": late_orders_by_state,
    "monthly_delivery": monthly_delivery,
    "customer_segments": customer_segments,
}
for name, df in gold_tables.items():
    rows = df.count()
    print(f"[CHECK] {name}: {rows:,} lignes")
    assert rows > 0, f"Table Gold vide : {name}"
assert customer_segments.count() == customer_segments.select("customer_unique_id").distinct().count()
assert category_performance.agg(F.sum("total_orders")).first()[0] == silver_count
assert late_orders_by_state.agg(F.sum("total_orders")).first()[0] == silver_count
print("Contrôles Gold validés.")

## 9 — Écriture Delta des quatre tables Gold

L’écriture utilise `overwrite` sur les **seuls chemins Gold ciblés**. Les données Bronze et Silver ne sont pas modifiées.

In [ ]:
for name, df in gold_tables.items():
    path = f"{GOLD_ROOT}/{name}"
    (df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(path))
    print(f"[GOLD OK] {path}")

## 10 — Relecture, validation et requêtes SQL

In [ ]:
for name in gold_tables:
    path = f"{GOLD_ROOT}/{name}"
    saved = spark.read.format("delta").load(path)
    print(f"[DELTA RELU] {name}: {saved.count():,} lignes")
    saved.createOrReplaceTempView(f"gold_{name}")

spark.sql("""
SELECT product_category, total_orders, total_revenue
FROM gold_category_performance
ORDER BY total_revenue DESC
LIMIT 10
""").show(truncate=False)

print("J3 terminé : quatre tables Gold Delta disponibles.")

## 11 — Historique Delta et contrôle final (facultatif)

In [ ]:
from delta.tables import DeltaTable
for name in gold_tables:
    path = f"{GOLD_ROOT}/{name}"
    print("Historique :", name)
    DeltaTable.forPath(spark, path).history(3).select("version", "timestamp", "operation").show(truncate=False)

print("J3 prêt pour J4 / dashboard Streamlit.")